# DRStageNet | Pré-processamento de retinografias APTOS 2019

**Objetivo:** reproduzir as etapas de preparação de imagens descritas por Men et al. (2025), sem incorporar CLAHE, filtros ou técnicas de outros artigos. O notebook preserva os cinco graus originais de retinopatia diabética e a divisão de dados existente no `APTOS.zip`.

**Referência:** Men, Y. et al. *Deep learning generalization for diabetic retinopathy staging from fundus images*. Physiological Measurement, 46, 015001, 2025. [DOI: 10.1088/1361-6579/ada86a](https://doi.org/10.1088/1361-6579/ada86a). Seções 2.2 e 2.3. [Texto consultado](https://www.researchgate.net/publication/387883048_Deep_learning_generalization_for_diabetic_retinopathy_staging_from_fundus_images).

Este notebook não treina o DRStageNet nem reproduz o experimento com seis bases. Ele prepara as imagens de uma base real para uso posterior em modelos de aprendizado profundo. Uso acadêmico, não diagnóstico.

## 1. Metodologia e escolhas de implementação

| Etapa | Descrição | Origem |
|---|---|---|
| Recorte | Retirar regiões pretas horizontais | Artigo, seção 2.2 |
| Padding | Completar o eixo menor até formar um quadrado, sem esticar a imagem | Artigo, seção 2.2 |
| Resize | 518 × 518, interpolação bilinear | Artigo, seção 2.2 |
| Aumento de dados | Inversão horizontal, variação de contraste, saturação e matiz, só no treino | Artigo, seção 2.3 |
| Tensor | RGB, float32 e normalização ImageNet do DINOv2 | Adaptação para a arquitetura citada, [código DINOv2](https://github.com/facebookresearch/dinov2/blob/main/dinov2/data/transforms.py) |

O texto não especifica o algoritmo de detecção das bordas pretas, o valor do limiar, o posicionamento exato do padding, a biblioteca de resize ou as amplitudes de aumento de dados. Nesta implementação, o recorte atua no eixo horizontal (colunas laterais), com limiar conservador de 5/255, padding preto centralizado e bilinear do Pillow (com redução antialias). Aumentos: contraste e saturação ±10%, matiz ±0,02 volta e inversão horizontal com probabilidade 0,5. São escolhas explícitas da implementação, não valores atribuídos ao artigo. O eixo de recorte pode ser ajustado se a inspeção mostrar que as bordas pretas estão acima e abaixo. A normalização ImageNet é fixa, não estimada a partir do APTOS; a seção 2.2 não especifica os valores usados pelos autores.

O método preserva a proporção, mas a redução de resolução ainda pode eliminar detalhes. Não há realce fotométrico nem filtragem de ruído obrigatória no artigo. O recorte não segmenta a retina nem garante preservação de lesões.

## 2. Preparar o dataset no Google Colab

1. No Google Drive, envie `APTOS.zip` para `Meu Drive/datasets/` (se já enviou, não repita o upload).
2. Abra este notebook no [Google Colab](https://colab.research.google.com/).
3. Execute a configuração e a montagem do Drive. Se o ZIP estiver em outro lugar, altere apenas `ZIP_DRIVE`.
4. Execute as demais células em ordem. A extração usa o disco local do Colab, com barra de progresso e reaproveitamento de uma extração concluída na mesma sessão.

A estrutura esperada é `train_1.csv`, `valid.csv`, `test.csv`, `train_images`, `val_images`, `test_images`. Pastas repetidas como `train_images/train_images` são aceitas. O ZIP original não será modificado. Não usamos `files.upload()` para o dataset, pois o arquivo grande já está no Drive.

In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, shutil, zipfile, sys, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageEnhance
from IPython.display import display
from tqdm.auto import tqdm

SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)
ORIGEM = "drive"  # "drive" ou "local"
ZIP_DRIVE = "/content/drive/MyDrive/datasets/APTOS.zip"
ZIP_LOCAL = "/content/APTOS.zip"
TRABALHO = Path("/content/aptos_drstagenet")
TRABALHO.mkdir(parents=True, exist_ok=True)
COLUNA_ID = None  # Ex.: "id_code"
COLUNA_ROTULO = None  # Ex.: "diagnosis"
TESTE_ROTULADO = None  # None: detectar; False: ignorar rótulos fictícios
TAMANHO = 518
LIMIAR_PRETO = 5  # escolha desta implementação
EIXO_RECORTE = "colunas"  # "linhas" se as faixas estiverem acima/abaixo
PROB_FLIP = 0.5
JITTER_CONTRASTE = 0.10
JITTER_SATURACAO = 0.10
JITTER_MATIZ = 0.02
MEDIA = np.array([0.485, 0.456, 0.406], dtype=np.float32)
DESVIO = np.array([0.229, 0.224, 0.225], dtype=np.float32)
EXTENSOES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

In [ ]:
if ORIGEM == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP = Path(ZIP_DRIVE)
elif ORIGEM == "local":
    ZIP = Path(ZIP_LOCAL)
else:
    raise ValueError("ORIGEM deve ser drive ou local.")
if not ZIP.is_file():
    raise FileNotFoundError(f"ZIP não encontrado: {ZIP}. Confira a configuração.")
if not zipfile.is_zipfile(ZIP):
    raise ValueError(f"ZIP inválido: {ZIP}")
print(f"Arquivo: {ZIP} | {ZIP.stat().st_size / 1024**2:.1f} MiB")

## 3. Extrair o ZIP e localizar os três conjuntos

O código seleciona a pasta que contém as imagens, mesmo quando há duas pastas com o mesmo nome aninhadas. Caso existam duas pastas independentes com imagens, interrompe para evitar misturar conjuntos.

In [ ]:
DIVISOES = {"treino": ("train_1.csv", "train_images"),
            "validacao": ("valid.csv", "val_images"),
            "teste": ("test.csv", "test_images")}

def extrair_zip(arquivo, destino):
    with zipfile.ZipFile(arquivo) as z:
        itens = z.infolist()
        if sum(i.file_size for i in itens) > shutil.disk_usage(destino).free:
            raise OSError("Espaço insuficiente para extrair o dataset.")
        for item in tqdm(itens, desc="Extraindo arquivos", unit="arquivo"):
            p = PurePosixPath(item.filename.replace("\\", "/"))
            if p.is_absolute() or ".." in p.parts or any(":" in s for s in p.parts):
                raise ValueError(f"Caminho inválido: {item.filename}")
            alvo = destino.joinpath(*p.parts)
            if item.is_dir():
                alvo.mkdir(parents=True, exist_ok=True)
            else:
                alvo.parent.mkdir(parents=True, exist_ok=True)
                with z.open(item) as entrada, alvo.open("wb") as saida:
                    shutil.copyfileobj(entrada, saida, length=1024**2)

def localizar_unico(base, nome, pasta=False):
    candidatos = [p for p in base.rglob("*") if p.name.lower() == nome.lower()
                  and (p.is_dir() if pasta else p.is_file())
                  and "__MACOSX" not in p.parts]
    if pasta and len(candidatos) > 1:
        diretos = [p for p in candidatos if any(f.is_file() and f.suffix.lower() in EXTENSOES for f in p.iterdir())]
        if diretos:
            candidatos = diretos
        else:
            # Se só houver subpastas por classe, escolhe o candidato mais interno.
            candidatos = [p for p in candidatos if not any(p in q.parents for q in candidatos if q != p)]
    if len(candidatos) != 1:
        raise ValueError(f"Esperado um item {nome}; encontrados: {candidatos}")
    return candidatos[0]

# Identifica uma extração completa na sessão por caminho, tamanho e modificação do ZIP.
stat = ZIP.stat()
chave = hashlib.sha256(f"{ZIP.resolve()}|{stat.st_size}|{stat.st_mtime_ns}".encode()).hexdigest()[:16]
DATASET_DIR = TRABALHO / f"extraido_{chave}"
MARCADOR = DATASET_DIR / ".extracao_concluida"
if not MARCADOR.exists():
    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)  # remove somente extração incompleta deste ZIP
    DATASET_DIR.mkdir()
    extrair_zip(ZIP, DATASET_DIR)
    MARCADOR.touch()
else:
    print("Reaproveitando extração concluída.")
CAMINHOS = {nome: (localizar_unico(DATASET_DIR, csv), localizar_unico(DATASET_DIR, pasta, True))
            for nome, (csv, pasta) in DIVISOES.items()}
for nome, (csv, pasta) in CAMINHOS.items():
    print(nome, "|", csv.name, "|", pasta)

## 4. Ler metadados e associar imagens

As colunas mais comuns são identificadas automaticamente. Caso seu CSV use outro nome, ajuste `COLUNA_ID` ou `COLUNA_ROTULO` na configuração. Mantemos os graus ICDR de 0 a 4, sem agrupamento binário, conforme a tarefa de estadiamento do DRStageNet. O teste pode não ter rótulos.

In [ ]:
TABELAS_BRUTAS = {}
for nome, (csv, pasta) in CAMINHOS.items():
    # Detecta vírgula ou ponto e vírgula; preserva identificadores como texto.
    with csv.open(encoding="utf-8-sig") as f:
        cabecalho = f.readline()
    separador = ";" if ";" in cabecalho and "," not in cabecalho else ("\t" if "\t" in cabecalho and "," not in cabecalho else ",")
    df = pd.read_csv(csv, sep=separador, dtype=str, encoding="utf-8-sig")
    df.columns = df.columns.str.strip()
    TABELAS_BRUTAS[nome] = df
    print(f"\n{nome}: {len(df)} registros | colunas: {df.columns.tolist()}")
    display(df.head(3))

In [ ]:
EXTENSOES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
IDS = ("id_code", "image_id", "image name", "image_name", "id", "image", "images", "filename", "file_name", "arquivo", "path", "image_path")
ROTULOS = ("diagnosis", "label", "labels", "grade", "level", "class", "target", "codigo_diagnostico", "severity", "dr_grade", "retinopathy_grade", "diagnostic")

def escolher_coluna(df, preferida, alternativas, obrigatoria=True):
    mapa = {str(col).strip().lower(): col for col in df.columns}
    if preferida is not None:
        if preferida.strip().lower() not in mapa:
            if obrigatoria:
                raise ValueError(f"Coluna {preferida} ausente: {df.columns.tolist()}")
            return None
        return mapa[preferida.strip().lower()]
    for opcao in alternativas:
        if opcao in mapa:
            return mapa[opcao]
    if obrigatoria:
        raise ValueError(f"Coluna não identificada. Colunas disponíveis: {df.columns.tolist()}")
    return None

def ler_conjunto(nome, df, pasta):
    col_id = escolher_coluna(df, COLUNA_ID, IDS)
    exige_rotulo = nome != "teste" or TESTE_ROTULADO is True
    col_y = (None if nome == "teste" and TESTE_ROTULADO is False else
             escolher_coluna(df, COLUNA_ROTULO, ROTULOS, exige_rotulo))
    if df.empty or df[col_id].isna().any():
        raise ValueError(f"{nome}: CSV vazio ou identificador ausente.")
    ids = df[col_id].str.strip().str.replace("\\", "/", regex=False)
    if (ids == "").any():
        raise ValueError(f"{nome}: identificador vazio.")
    # Indexação por nome e por stem para aceitar IDs com ou sem extensão.
    por_nome, por_id = {}, {}
    for img in pasta.rglob("*"):
        if img.is_file() and img.suffix.lower() in EXTENSOES:
            por_nome.setdefault(img.name.lower(), []).append(img)
            por_id.setdefault(img.stem.lower(), []).append(img)
    caminhos = []
    for identificador in ids:
        arquivo = PurePosixPath(identificador).name
        p = Path(arquivo)
        opcoes = (por_nome.get(arquivo.lower(), []) if p.suffix.lower() in EXTENSOES
                  else por_id.get(arquivo.lower(), []))
        if len(opcoes) != 1:
            raise ValueError(f"{nome}: imagem ausente ou ambígua: {identificador} ({len(opcoes)} opções)")
        caminhos.append(opcoes[0])
    tabela = pd.DataFrame({"id_code": [p.stem for p in caminhos],
                           "caminho": [str(p) for p in caminhos], "conjunto": nome})
    if tabela["id_code"].str.lower().duplicated().any():
        raise ValueError(f"{nome}: imagem repetida no CSV.")
    if col_y is not None:
        graus = pd.to_numeric(df[col_y], errors="coerce")
        if not graus.isin([0, 1, 2, 3, 4]).all():
            raise ValueError(f"{nome}: rótulos inválidos na coluna {col_y}. Esperados graus inteiros de 0 a 4. Se o teste tiver rótulos fictícios, configure TESTE_ROTULADO = False.")
        tabela["diagnosis"] = graus.astype("int32").to_numpy()
    else:
        tabela["diagnosis"] = pd.Series(pd.NA, index=tabela.index, dtype="Int32")
    print(f"{nome}: {len(tabela)} imagens associadas; {sum(map(len, por_id.values()))} arquivos na pasta; ID={col_id}; rótulo={col_y}")
    return tabela

conjuntos = {nome: ler_conjunto(nome, TABELAS_BRUTAS[nome], pasta)
             for nome, (_, pasta) in CAMINHOS.items()}
treino, validacao, teste = (conjuntos[n] for n in DIVISOES)

# Não permite reutilização de identificadores entre as três divisões.
ids_repetidos = []
for a, b in [("treino", "validacao"), ("treino", "teste"), ("validacao", "teste")]:
    repetidos = set(conjuntos[a].id_code.str.lower()) & set(conjuntos[b].id_code.str.lower())
    if repetidos:
        ids_repetidos.extend((a, b, i) for i in sorted(repetidos))
        print(f"ATENÇÃO: IDs repetidos entre {a} e {b}: {sorted(repetidos)[:10]}")
if not ids_repetidos:
    print("Nenhum identificador repetido entre os conjuntos.")
print("A checagem por ID não exclui duplicatas de conteúdo com nomes diferentes.")
amostras = treino  # Exemplos visuais usam apenas imagens de treino.
display(treino.head())

In [ ]:
resumo = pd.DataFrame([{"conjunto": nome, "imagens": len(df),
                        "rotuladas": int(df.diagnosis.notna().sum())}
                       for nome, df in conjuntos.items()])
display(resumo)
rotulados = pd.concat([treino, validacao], ignore_index=True)
contagem = pd.crosstab(rotulados.diagnosis, rotulados.conjunto).reindex(range(5), fill_value=0)
display(contagem)
contagem.plot.bar(rot=0, figsize=(8, 4), ylabel="Quantidade", xlabel="Grau de RD")
plt.tight_layout(); plt.show()

## 5. Auditoria de duplicatas entre conjuntos

O hash SHA-256 identifica arquivos com bytes idênticos, inclusive com nomes diferentes. O relatório não elimina imagens nem altera a divisão. Se houver sobreposição, revise as divisões antes do treinamento. Duplicatas recomprimidas, imagens quase idênticas e pacientes compartilhados não são excluídos por esta verificação. Sem IDs de paciente, não é possível assegurar independência por paciente. Esta auditoria não ajusta parâmetros de pré-processamento sobre a validação ou o teste.

In [ ]:
def hash_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(1024**2), b""):
            h.update(bloco)
    return h.hexdigest()

auditoria = pd.concat(conjuntos.values(), ignore_index=True)
auditoria["sha256"] = [hash_arquivo(p) for p in tqdm(auditoria.caminho, desc="Auditando arquivos")]
grupos = auditoria.groupby("sha256").conjunto.nunique()
conflitos = auditoria[auditoria.sha256.isin(grupos[grupos > 1].index)].sort_values("sha256")
conflitos.to_csv(TRABALHO / "duplicatas_entre_conjuntos.csv", index=False)
print("Grupos de arquivos idênticos em conjuntos diferentes:", int((grupos > 1).sum()))
if not conflitos.empty:
    display(conflitos.head(20))
    print("ATENÇÃO: não use estas divisões para avaliar o modelo antes de corrigir o vazamento.")
else:
    print("Nenhuma duplicata exata entre conjuntos encontrada.")

## 6. Implementar o pré-processamento original

Primeiro, removemos somente colunas totalmente escuras nas extremidades do eixo horizontal; se o eixo estiver configurado como linhas, removemos linhas externas. O limiar é fixo em todas as divisões. Uma coluna com qualquer pixel acima do limiar permanece, o que torna a operação conservadora. Não removemos faixas escuras internas. Depois, usamos padding preto centralizado, sem mudar a relação largura/altura do conteúdo. O quadrado é redimensionado uma única vez com bilinear. O arredondamento e a interpolação podem alterar pixels, mas não esticam um eixo em relação ao outro.

In [ ]:
def carregar_rgb(caminho):
    with Image.open(caminho) as im:
        return im.convert("RGB")

def recortar_faixas_pretas(imagem, limiar=LIMIAR_PRETO, eixo=EIXO_RECORTE):
    a = np.asarray(imagem)
    pixels_validos = np.max(a, axis=2) > limiar
    if eixo == "colunas":
        validos = np.flatnonzero(pixels_validos.any(axis=0))
        if validos.size == 0:
            raise ValueError("Imagem inteiramente escura; revisar arquivo/limiar.")
        caixa = (int(validos[0]), 0, int(validos[-1])+1, imagem.height)
    elif eixo == "linhas":
        validos = np.flatnonzero(pixels_validos.any(axis=1))
        if validos.size == 0:
            raise ValueError("Imagem inteiramente escura; revisar arquivo/limiar.")
        caixa = (0, int(validos[0]), imagem.width, int(validos[-1])+1)
    else:
        raise ValueError("EIXO_RECORTE deve ser colunas ou linhas.")
    return imagem.crop(caixa), caixa

def completar_quadrado(imagem):
    w, h = imagem.size
    lado = max(w, h)
    esquerda, topo = (lado-w)//2, (lado-h)//2
    padding = (esquerda, topo, lado-w-esquerda, lado-h-topo)
    return ImageOps.expand(imagem, border=padding, fill=(0,0,0)), padding

def preprocessar(caminho, retornar_etapas=False):
    original = carregar_rgb(caminho)
    recorte, caixa = recortar_faixas_pretas(original)
    quadrado, padding = completar_quadrado(recorte)
    final = quadrado.resize((TAMANHO, TAMANHO), Image.Resampling.BILINEAR)
    if retornar_etapas:
        return original, recorte, quadrado, final, caixa, padding
    return final

def normalizar(imagem):
    rgb = np.asarray(imagem, dtype=np.float32) / np.float32(255.0)
    return ((rgb - MEDIA) / DESVIO).astype(np.float32)

## 7. Visualização por etapa

Exemplos exclusivamente do treino, um por grau disponível. As imagens exibidas são RGB antes da normalização estatística. A última coluna apresenta a imagem que será entregue ao modelo após conversão para tensor.

In [ ]:
exemplos = treino.groupby("diagnosis", group_keys=False).sample(n=1, random_state=SEMENTE).sort_values("diagnosis")
fig, eixos = plt.subplots(len(exemplos), 4, figsize=(15, 3.8*len(exemplos)), squeeze=False)
registros = []
for i, linha in enumerate(exemplos.itertuples(index=False)):
    original, recorte, quadrado, final, caixa, padding = preprocessar(linha.caminho, True)
    for ax, im, nome in zip(eixos[i], [original, recorte, quadrado, final],
                            ["Original", "Recorte", "Padding quadrado", "Bilinear"]):
        ax.imshow(im); ax.set_title(f"{nome} | {im.width}×{im.height}"); ax.axis("off")
    eixos[i, 0].set_title(f"Grau {linha.diagnosis} | Original\n{original.width}×{original.height}")
    registros.append({"id": linha.id_code, "grau": linha.diagnosis,
                      "original": original.size, "recorte": recorte.size,
                      "padding": padding, "final": final.size,
                      "area_removida_pct": round(100*(1-recorte.width*recorte.height/(original.width*original.height)), 2)})
plt.tight_layout(); plt.show()
display(pd.DataFrame(registros))

## 8. Verificações geométricas e numéricas

O teste verifica se o recorte é uma cópia exata da região selecionada e se o padding contém o recorte sem modificação. A transformação final usa o mesmo fator de escala nos dois eixos. Não é uma avaliação clínica da preservação de microlesões.

In [ ]:
for linha in exemplos.itertuples(index=False):
    original, recorte, quadrado, final, caixa, padding = preprocessar(linha.caminho, True)
    esq, topo, dir_, base = padding
    assert np.array_equal(np.asarray(recorte), np.asarray(original.crop(caixa)))
    assert np.array_equal(np.asarray(recorte), np.asarray(quadrado.crop((esq, topo, esq+recorte.width, topo+recorte.height))))
    assert quadrado.width == quadrado.height == max(recorte.size)
    assert final.size == (518, 518)
    tensor = normalizar(final)
    assert tensor.shape == (518, 518, 3) and tensor.dtype == np.float32
    assert np.isfinite(tensor).all()
    assert np.array_equal(np.asarray(final), np.asarray(preprocessar(linha.caminho)))
    print(f"{linha.id_code}: verificações OK | escala uniforme {TAMANHO/quadrado.width:.4f}")
print("Tensor HWC:", tensor.shape, "| faixa:", float(tensor.min()), float(tensor.max()))

## 9. Aumento de dados somente no treinamento

O artigo descreve inversão horizontal e variações de contraste, saturação e matiz, sem divulgar amplitudes. Os valores da configuração são conservadores e explícitos. Não adicionamos rotação, zoom, MixUp ou CLAHE. A operação recebe uma imagem RGB preparada, modifica suas cores e pode invertê-la horizontalmente, mantendo o grau de RD. Validação, teste e inferência não recebem aumento de dados. As imagens exportadas na seção 11 também não são aumentadas.

In [ ]:
def aumentar(imagem, rng):
    im = imagem.copy()
    if rng.random() < PROB_FLIP:
        im = ImageOps.mirror(im)
    im = ImageEnhance.Contrast(im).enhance(float(rng.uniform(1-JITTER_CONTRASTE, 1+JITTER_CONTRASTE)))
    im = ImageEnhance.Color(im).enhance(float(rng.uniform(1-JITTER_SATURACAO, 1+JITTER_SATURACAO)))
    hsv = np.array(im.convert("HSV"), dtype=np.uint8)
    delta = int(round(256*rng.uniform(-JITTER_MATIZ, JITTER_MATIZ)))
    hsv[..., 0] = ((hsv[..., 0].astype(np.int16) + delta) % 256).astype(np.uint8)
    return Image.fromarray(hsv, mode="HSV").convert("RGB")

base = preprocessar(treino.iloc[0].caminho)
rng = np.random.default_rng(SEMENTE)
fig, eixos = plt.subplots(1, 5, figsize=(16, 4))
for i, im in enumerate([base] + [aumentar(base, rng) for _ in range(4)]):
    eixos[i].imshow(im); eixos[i].set_title("Sem aumento" if i == 0 else f"Variação {i}"); eixos[i].axis("off")
plt.tight_layout(); plt.show()

## 10. Preparação para o modelo com PyTorch (opcional)

O DINOv2 recebe um tensor float32 na ordem C × H × W. Usamos as constantes ImageNet publicadas no código do DINOv2 e não estimamos estatísticas a partir de validação ou teste. Os pixels visualizáveis continuam em RGB; o tensor normalizado pode conter valores fora de [0, 1]. O dataset carrega imagens sob demanda, sem manter todos os tensores em RAM. Não há treinamento nesta seção. Se houver rótulos ausentes no teste, retorna apenas imagem e ID; caso contrário, retorna também o grau.

Se a auditoria encontrar sobreposição entre conjuntos, esta seção bloqueia a criação dos lotes até que o problema seja corrigido. As etapas anteriores continuam disponíveis para inspeção.

In [ ]:
PREPARAR_LOTES = False  # True para preparar lotes PyTorch
BATCH_SIZE = 8  # escolha para o Colab, não é o batch do artigo
if PREPARAR_LOTES:
    if ids_repetidos or not conflitos.empty:
        raise ValueError("Revise as duplicatas entre conjuntos antes de criar os lotes.")
    import torch
    from torch.utils.data import Dataset, DataLoader
    torch.manual_seed(SEMENTE)
    class DatasetRetina(Dataset):
        def __init__(self, tabela, treinamento=False):
            self.tabela = tabela.reset_index(drop=True)
            self.treinamento = treinamento
            self.rotulado = self.tabela.diagnosis.notna().all()
            self.rng = np.random.default_rng(SEMENTE)
        def __len__(self):
            return len(self.tabela)
        def __getitem__(self, idx):
            linha = self.tabela.iloc[idx]
            im = preprocessar(linha.caminho)
            if self.treinamento:
                im = aumentar(im, self.rng)
            x = torch.from_numpy(normalizar(im).transpose(2,0,1).copy())
            item = {"imagem": x, "id": linha.id_code}
            if self.rotulado:
                item["grau"] = torch.tensor(int(linha.diagnosis), dtype=torch.long)
            return item
    lotes = {nome: DataLoader(DatasetRetina(df, nome == "treino"), batch_size=BATCH_SIZE,
                               shuffle=nome == "treino", num_workers=0,
                               generator=torch.Generator().manual_seed(SEMENTE))
             for nome, df in conjuntos.items()}
    for nome, loader in lotes.items():
        lote = next(iter(loader))
        print(nome, lote["imagem"].shape, lote["imagem"].dtype, "graus" if "grau" in lote else "sem rótulos")
else:
    print("Lotes desativados. Altere PREPARAR_LOTES se precisar.")

## 11. Exportar imagens processadas para o Drive (opcional)

A exportação preserva o RGB de 8 bits em PNG sem perdas adicionais de compressão. Os arquivos originais permanecem intactos. Os CSVs registram o grau original e os caminhos relativos dos PNGs. A exportação não contém tensores normalizados nem imagens aumentadas, pois estas operações devem ser feitas no carregamento do modelo. Mantenha a auditoria de duplicatas junto ao dataset para revisão. O ZIP de saída é criado no disco local e copiado para o Drive, evitando download direto pelo navegador. Por padrão a exportação fica desativada; `MAXIMO_POR_CONJUNTO = None` exporta todos os registros.

In [ ]:
EXPORTAR = False
MAXIMO_POR_CONJUNTO = None
DESTINO_DRIVE = Path("/content/drive/MyDrive/datasets/APTOS_DRStageNet_518.zip")
if EXPORTAR:
    import tempfile
    pasta_saida = Path(tempfile.mkdtemp(prefix="export_", dir=TRABALHO))
    for nome, df in conjuntos.items():
        pasta = pasta_saida / nome
        pasta.mkdir()
        selecionados = df if MAXIMO_POR_CONJUNTO is None else df.head(MAXIMO_POR_CONJUNTO)
        linhas = []
        for linha in tqdm(selecionados.itertuples(index=False), total=len(selecionados), desc=nome):
            arquivo = f"{linha.id_code}.png"
            preprocessar(linha.caminho).save(pasta / arquivo, format="PNG")
            linhas.append({"id_code": linha.id_code, "diagnosis": linha.diagnosis,
                           "arquivo": f"{nome}/{arquivo}"})
        pd.DataFrame(linhas, columns=["id_code", "diagnosis", "arquivo"]).to_csv(pasta_saida / f"{nome}.csv", index=False)
    conflitos.drop(columns="caminho").to_csv(pasta_saida / "duplicatas_entre_conjuntos.csv", index=False)
    configuracao = {"tamanho": TAMANHO, "eixo_recorte": EIXO_RECORTE, "limiar_preto": LIMIAR_PRETO,
                    "interpolacao": "Pillow BILINEAR", "padding": "preto centralizado",
                    "media_dinov2": MEDIA.tolist(), "desvio_dinov2": DESVIO.tolist(),
                    "aumento_exportado": False, "maximo_por_conjunto": MAXIMO_POR_CONJUNTO}
    (pasta_saida / "configuracao.json").write_text(json.dumps(configuracao, indent=2), encoding="utf-8")
    caminho_zip = Path(shutil.make_archive(str(pasta_saida), "zip", root_dir=pasta_saida))
    if ORIGEM == "drive":
        if DESTINO_DRIVE.exists():
            raise FileExistsError(f"Já existe {DESTINO_DRIVE}. Altere o nome de saída para não sobrescrever.")
        DESTINO_DRIVE.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(caminho_zip, DESTINO_DRIVE)
        print("ZIP no Drive:", DESTINO_DRIVE)
    else:
        print("ZIP local:", caminho_zip)
else:
    print("Exportação desativada. Altere EXPORTAR para True se desejar salvar.")

## 12. Limites e referência

O notebook reproduz as operações geométricas descritas na seção 2.2 e os tipos de aumento da seção 2.3. As decisões não especificadas pelos autores foram documentadas. Não aplicamos correção de iluminação ou filtros adicionais. A integridade geométrica pode ser conferida, mas a preservação de lesões pequenas precisa de avaliação com anotações de lesões, e a generalização exige avaliação em bases externas. A auditoria por hash não garante ausência de vazamento por paciente ou de duplicatas aproximadas.

- Men et al. (2025). [Deep learning generalization for diabetic retinopathy staging from fundus images](https://doi.org/10.1088/1361-6579/ada86a).
- [Código de normalização DINOv2](https://github.com/facebookresearch/dinov2/blob/main/dinov2/data/transforms.py).
- [APTOS 2019 Blindness Detection](https://www.kaggle.com/competitions/aptos2019-blindness-detection/data).